In [1]:
import polars as pl

pl.Config.set_tbl_rows(1000)    
pl.Config.set_tbl_cols(20)      
pl.Config.set_fmt_str_lengths(50)

polars.config.Config

In [2]:

CREAMY = r'(?i)сливочный|cream\s*cheese|кремовый'
CURD = r'(?i)творожный'
MELTED = r'(?i)плавленый'
VANILLA = r'(?i)ванильный'
CREAMY_CURD_MELTED = r'(?i)сливочный|cream\s*cheese|кремовый|творожный|плавленый'

CHEESE_KEYWORDS = [
    'сыр сливочный',
    'сыр кремовый',
    'сыр твердый',
    'сыр плавленый',
    'сыр творожный',
    'сыр полутвердый',
    'cream cheese',
    'cheese ',
    'сыр '
]

CHEESE_TYPE_PATTERN = r'(?i)(' + '|'.join(CHEESE_KEYWORDS) + r')'
WEIGHT_PATTERN = r'(\d+)\s+[гg]'
FAT_PERCENTAGE_PATTERN = r'(\d+)\s*%'
RAW_TEXT_PATTERN = rf'{CHEESE_TYPE_PATTERN}\s*([^\d\n]+)'

CATEGORY_PATTERNS = {
    'молочка': r'(?i)\b(?:молоч|кефир|сметан|творог|йогурт|сливк|сгущен|морожен|пахт[аы]|обрат|айран|тан|кумыс)\b',
    'мясная продукция': r'(?i)\b(?:мяс|говяж|свинин|куриц|индейк|фарш|колбас|сосиск|ветчин|бекон|шашлык|котлет)\b',
    'рыба и морепродукты': r'(?i)\b(?:рыб|лосос|семг|форел|треск|кревет|кальмар|икр|краб|миди|тунец|скумбр|сельд)\b',
    'выпечка': r'(?i)\b(?:хлеб|батон|булк|пирог|печень|торт|кекс|булоч|круассан|лаваш|печен[ияе])\b',
    'напитки': r'(?i)\b(?:чай|кофе|вода|сок|компот|квас|лимонад|кола|энергетик|минеральн|газирован)\b',
    'соус': r'(?i)\b(?:соус|кетчуп|майонез|аджик|пест[оы]|ткемал|терияки|барбекю)\b',
    'бытовая химия': r'(?i)\b(?:порош|гель|средств|чистящ|моющ|отбелив|пятновыв|антижир|шуманит|доместос|фейри)\b',
    'личная гигиена': r'(?i)\b(?:мыл|шампун|паст|щетк|дезодор|крем|прокладк|подгузник|салфет|гигиен)\b',
    'овощи и Фрукты': r'(?i)\b(?:яблок|груш|банан|лимон|апельсин|помидор|огур|картоф|лук|чеснок|капуст|морков)\b',
    'услуга': r'(?i)\b(?:доставк|упаковк|сервис|сборк)\b',
}

In [3]:
df = pl.scan_csv('./top_cheese_test.tsv', separator='\t')

In [4]:

df_cheese = (
    df
    .with_columns(pl.col('title').str.extract(RAW_TEXT_PATTERN, 2).alias('raw_text'))
    .filter(pl.col('title').str.contains(CHEESE_TYPE_PATTERN))
)

raw_words_list = pl.col('raw_text').str.strip_chars().str.split(' ')
words_count = raw_words_list.list.len()
mid_idx = pl.max_horizontal(words_count // 2 - 1, 1)

normalized_df_cheese = (
    df_cheese.with_columns(
        cheese_type=pl.when(pl.col('title').str.contains(CREAMY)).then(pl.lit('сыр сливочный'))
                .when(pl.col('title').str.contains(VANILLA)).then(pl.lit('сыр ванильный'))
                .when(pl.col('title').str.contains(CURD)).then(pl.lit('сыр творожный'))
                .when(pl.col('title').str.contains(MELTED)).then(pl.lit('сыр плавленый'))
                .otherwise(pl.lit('сыр обычный')),
        
        is_canceled=pl.col('title').str.contains(r'(?i)возмещение'),
        weight=pl.col('title').str.extract(WEIGHT_PATTERN, 1),
        fat_percentage=pl.col('title').str.extract(FAT_PERCENTAGE_PATTERN, 1),

        brand=pl.when(pl.col('title').str.contains(CREAMY_CURD_MELTED))
                .then(raw_words_list.list.slice(0, mid_idx).list.join(' '))
                .otherwise(pl.col('raw_text')),
                    
        flavor=pl.when(pl.col('title').str.contains(CREAMY_CURD_MELTED) & (words_count > 1))
                .then(raw_words_list.list.slice(mid_idx).list.join(' '))
                .otherwise(None),
    )
).drop('raw_text')

normalized_df_cheese.collect().head(10)

user_id,id,date_actual,title,sum,quantity,is_mm,check_id,cheese_type,is_canceled,weight,fat_percentage,brand,flavor
i64,str,str,str,f64,f64,bool,str,str,bool,str,str,str,str
85633,"""0bad79b1-5372-4409-abb5-46b4a792e5a7""","""2023-04-07 12:45:26.000""","""сыр плавленый карат дружба 45% бзмж 400 г""",156.58,1.0,true,"""7b42ddcd-2ec3-4a21-b059-987e4644a3da""","""сыр плавленый""",false,"""400""","""45""","""карат""","""дружба"""
85633,"""1578dc2f-4ee7-4035-b87e-21d44a06c6ec""","""2023-01-26 09:48:48.000""","""сыр творожный hochland сливочный 60% 140 г""",119.99,1.0,false,"""4c78f077-a828-439a-90ac-04746211b232""","""сыр сливочный""",false,"""140""","""60""","""hochland""","""сливочный"""
85633,"""D5JvcI0BIXZkW_HSgS5I""","""2024-02-02 09:38:22.000""","""сыр творожный hochland сливочный 60% 140 г""",62.26,1.0,true,"""9289f2f7-b8fd-4dbb-93d9-25fef93c9c31""","""сыр сливочный""",false,"""140""","""60""","""hochland""","""сливочный"""
85633,"""512e8175-fb6c-4ff0-ade6-064cff7a4417""","""2023-07-15 11:20:14.000""","""возмещение за: сыр плавленый hochland сливочный 55…",242.26,1.0,false,"""3271d338-8247-4ab7-a76f-5aa4211ebd6f""","""сыр сливочный""",true,"""400""","""55""","""hochland""","""сливочный"""
85633,"""vVMoBosBCTUrFDcdfiRu""","""2023-10-05 10:35:01.000""","""возмещение за: сыр творожный hochland сливочный 60…",139.7,1.0,true,"""4aea2c39-8736-4bae-b02b-3672fdb8a424""","""сыр сливочный""",true,"""220""","""60""","""hochland""","""сливочный"""
85633,"""fWV8B4wBuO4rswBD3Yl_""","""2023-11-24 18:00:50.000""","""сыр творожный hochland сливочный 60% 140 г""",119.9,1.0,true,"""add49254-1017-471c-89b6-9dc372a0d82a""","""сыр сливочный""",false,"""140""","""60""","""hochland""","""сливочный"""
85633,"""1d9f8a38-429b-4174-8ea1-ba7bb7cfa185""","""2023-08-18 09:22:00.000""","""сыр творожный hochland с зеленью 60% 220 г""",139.99,1.0,true,"""a8dd8591-8d6c-4bea-800a-0426ad78e2d5""","""сыр творожный""",false,"""220""","""60""","""hochland""","""с зеленью"""
85633,"""8b6d012e-64d3-4147-b7f2-ed49c2fc9799""","""2023-07-19 16:02:57.000""","""возмещение за: сыр творожный hochland сливочный 60…",119.99,1.0,true,"""7fa1af37-a55c-4806-a8e6-abe960ecefdd""","""сыр сливочный""",true,"""140""","""60""","""hochland""","""сливочный"""
85633,"""-mqyGosBMQnv8mcBHzFD""","""2023-10-09 11:37:25.000""","""возмещение за: сыр творожный hochland сливочный 60…",139.39,1.0,true,"""b4036f21-eab0-4d54-9d6d-ecd60006bb4c""","""сыр сливочный""",true,"""220""","""60""","""hochland""","""сливочный"""


In [5]:

df_other_products = df.filter(~pl.col('title').str.contains(CHEESE_TYPE_PATTERN))

category_expr = pl.lit('другое')
for cat_name, pattern in CATEGORY_PATTERNS.items():
    category_expr = pl.when(pl.col('title').str.contains(pattern)).then(pl.lit(cat_name)).otherwise(category_expr)

result = (
    df_other_products.with_columns(
        category=category_expr,
        is_canceled=pl.col('title').str.contains(r'(?i)возмещение'),
    )
)

result.filter(~pl.col('category').str.contains('другое')).collect().head(10)

user_id,id,date_actual,title,sum,quantity,is_mm,check_id,category,is_canceled
i64,str,str,str,f64,f64,bool,str,str,bool
85633,"""8aedd593-6b52-43d6-8241-135ea59316bd""","""2023-05-25 09:43:45.000""","""возмещение за: лаваш армянский 290 г""",60.0,1.0,true,"""ecf280d8-7596-4a3d-8a4b-c3c84dd7ddcc""","""выпечка""",true
85633,"""c4479376-bb41-4fcc-9439-bb53368d2b27""","""2023-02-01 09:42:51.000""","""лаваш семь печей прямоугольный пшеничный 200 г""",49.99,1.0,false,"""2e6f7177-1ddf-427d-80ca-0aeaba27003e""","""выпечка""",false
85633,"""a06955c5-df01-42b6-b1d6-ca796ffa9965""","""2023-04-07 12:45:26.000""","""торт faretti смородиновый 400 г""",112.2,1.0,false,"""7b42ddcd-2ec3-4a21-b059-987e4644a3da""","""выпечка""",false
85633,"""6333aaf3-4911-4d1c-9fab-6b5511705585""","""2023-03-03 08:29:25.000""","""лук репчатый""",16.62024,0.546,true,"""616641e7-3d2e-4415-b097-9ebc92a9b1ac""","""овощи и Фрукты""",false
85633,"""36fd567b-d71f-482b-9493-14db02535963""","""2023-07-15 11:20:14.000""","""возмещение за: холодный чай lipton зеленый 0,5 л""",52.3,1.0,false,"""3271d338-8247-4ab7-a76f-5aa4211ebd6f""","""напитки""",true
85633,"""205e66da-b3a2-4b5a-84db-b31f688beb3e""","""2023-06-25 12:10:09.000""","""возмещение за: кетчуп махеевъ шашлычный 300 г""",34.51,1.0,false,"""f2ffe7be-5954-4063-8cc1-8d38566352e0""","""соус""",true
85633,"""7dc55306-c843-438f-a93e-ae3efff58f57""","""2023-09-05 15:11:51.000""","""возмещение за: батон семь печей нарезной пшеничный…",41.03,1.0,true,"""547bfcb6-9d44-45c1-902c-d68ed5cda548""","""выпечка""",true
85633,"""0bf27f2e-f923-4530-ae9c-3159fe338045""","""2023-03-03 10:08:32.000""","""лук репчатый""",16.62024,0.546,true,"""66c55226-2e31-4a39-8ab1-c307fd55c933""","""овощи и Фрукты""",false
85633,"""ec1395fb-baeb-448c-8041-a9fae978a4a9""","""2023-05-25 09:43:45.000""","""возмещение за: майонез mr.ricco на перепелином яйц…",72.65,1.0,true,"""ecf280d8-7596-4a3d-8a4b-c3c84dd7ddcc""","""соус""",true
